# Image Processing and Vision Course Project

## Image Distortions

This notebook applies controlled image distortions at several severity levels to the fixed experimental image subset.

In [ ]:
!pip install -q opencv-python matplotlib numpy pandas torch torchvision ultralytics

In [ ]:
import random

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from pathlib import Path
from torchvision.datasets import VOCSegmentation

In [ ]:
SEED = 10

random.seed(SEED)
np.random.seed(SEED)

print(f"Random seed set to {SEED}")

In [ ]:
# Repository-relative project paths
from pathlib import Path

PROJECT_ROOT = Path.cwd()

# Supports running either from the repository root or from the notebooks folder
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)
print("Figures directory:", FIGURES_DIR)


In [ ]:
# Load PASCAL VOC 2012 and the same fixed subset used in previous notebooks
dataset = VOCSegmentation(
    root=DATA_DIR,
    year="2012",
    image_set="train",
    download=True
)

selected_indices_path = RESULTS_DIR / "selected_indices.csv"

if selected_indices_path.exists():
    selected_indices_df = pd.read_csv(selected_indices_path)
    selected_indices = selected_indices_df["dataset_index"].tolist()
    print("Loaded fixed indices from:", selected_indices_path)
else:
    # Reproduce the exact same subset when the CSV is not available
    rng = np.random.default_rng(SEED)
    selected_indices = sorted(
        rng.choice(len(dataset), size=100, replace=False).tolist()
    )
    selected_indices_df = pd.DataFrame({
        "dataset_index": selected_indices
    })
    selected_indices_df.to_csv(selected_indices_path, index=False)
    print("Recreated and saved fixed indices to:", selected_indices_path)

print(f"Number of dataset samples: {len(dataset)}")
print(f"Loaded {len(selected_indices)} selected indices")
print("First 10 selected indices:", selected_indices[:10])


In [ ]:
# Add Gaussian noise to an RGB image
def add_gaussian_noise(image, sigma, seed=SEED):
    image_np = np.array(image.convert("RGB"), dtype=np.float32)

    rng = np.random.default_rng(seed)
    noise = rng.normal(
        loc=0.0,
        scale=sigma,
        size=image_np.shape
    )

    noisy_image = image_np + noise
    noisy_image = np.clip(noisy_image, 0, 255).astype(np.uint8)

    return Image.fromarray(noisy_image)

In [ ]:
# Visualize Gaussian noise at three severity levels
sample_index = selected_indices[0]
sample_image, _ = dataset[sample_index]

noise_levels = {
    "Mild": 10,
    "Medium": 25,
    "Severe": 50
}

fig, axes = plt.subplots(1, 4, figsize=(16, 5))

axes[0].imshow(sample_image)
axes[0].set_title("Original")
axes[0].axis("off")

for axis, (level_name, sigma) in zip(axes[1:], noise_levels.items()):
    noisy_image = add_gaussian_noise(
        sample_image,
        sigma=sigma,
        seed=SEED
    )

    axis.imshow(noisy_image)
    axis.set_title(f"{level_name}\nσ = {sigma}")
    axis.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Apply JPEG compression at a selected quality level
from io import BytesIO

def apply_jpeg_compression(image, quality):
    image_rgb = image.convert("RGB")

    buffer = BytesIO()
    image_rgb.save(
        buffer,
        format="JPEG",
        quality=quality
    )

    buffer.seek(0)

    compressed_image = Image.open(buffer).convert("RGB")

    return compressed_image

In [ ]:
# Visualize JPEG compression at three severity levels
jpeg_quality_levels = {
    "Mild": 50,
    "Medium": 20,
    "Severe": 5
}

fig, axes = plt.subplots(1, 4, figsize=(16, 5))

axes[0].imshow(sample_image)
axes[0].set_title("Original")
axes[0].axis("off")

for axis, (level_name, quality) in zip(axes[1:], jpeg_quality_levels.items()):
    compressed_image = apply_jpeg_compression(
        sample_image,
        quality=quality
    )

    axis.imshow(compressed_image)
    axis.set_title(f"{level_name}\nQuality = {quality}")
    axis.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Simulate low-light conditions by reducing image brightness
def apply_low_light(image, brightness_factor):
    image_np = np.array(image.convert("RGB"), dtype=np.float32)

    dark_image = image_np * brightness_factor
    dark_image = np.clip(dark_image, 0, 255).astype(np.uint8)

    return Image.fromarray(dark_image)


In [ ]:
# Visualize low-light distortion at three severity levels
low_light_levels = {
    "Mild": 0.6,
    "Medium": 0.35,
    "Severe": 0.15
}

fig, axes = plt.subplots(1, 4, figsize=(16, 5))

axes[0].imshow(sample_image)
axes[0].set_title("Original")
axes[0].axis("off")

for axis, (level_name, brightness_factor) in zip(
    axes[1:],
    low_light_levels.items()
):
    dark_image = apply_low_light(
        sample_image,
        brightness_factor=brightness_factor
    )

    axis.imshow(dark_image)
    axis.set_title(
        f"{level_name}\nBrightness = {brightness_factor}"
    )
    axis.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Define all distortion types and their severity levels
distortion_config = {
    "gaussian_noise": {
        "mild": 10,
        "medium": 25,
        "severe": 50
    },
    "jpeg_compression": {
        "mild": 50,
        "medium": 20,
        "severe": 5
    },
    "low_light": {
        "mild": 0.6,
        "medium": 0.35,
        "severe": 0.15
    }
}

print(distortion_config)

In [ ]:
# Create a summary table of the distortion settings
distortion_rows = []

for distortion_name, severity_dict in distortion_config.items():
    for severity_level, parameter_value in severity_dict.items():
        distortion_rows.append({
            "distortion_type": distortion_name,
            "severity_level": severity_level,
            "parameter_value": parameter_value
        })

distortion_summary_df = pd.DataFrame(distortion_rows)

display(distortion_summary_df)

In [ ]:
# Apply a selected distortion and severity level to an input image
def apply_distortion(image, distortion_type, severity_level, seed=SEED):
    if distortion_type not in distortion_config:
        raise ValueError(f"Unknown distortion type: {distortion_type}")

    if severity_level not in distortion_config[distortion_type]:
        raise ValueError(
            f"Unknown severity level '{severity_level}' "
            f"for distortion '{distortion_type}'"
        )

    parameter_value = distortion_config[distortion_type][severity_level]

    if distortion_type == "gaussian_noise":
        return add_gaussian_noise(
            image,
            sigma=parameter_value,
            seed=seed
        )

    if distortion_type == "jpeg_compression":
        return apply_jpeg_compression(
            image,
            quality=int(parameter_value)
        )

    if distortion_type == "low_light":
        return apply_low_light(
            image,
            brightness_factor=parameter_value
        )

    raise ValueError(f"Unsupported distortion type: {distortion_type}")

In [ ]:
# Test the unified distortion function
test_distorted_image = apply_distortion(
    sample_image,
    distortion_type="gaussian_noise",
    severity_level="medium"
)

plt.figure(figsize=(8, 5))
plt.imshow(test_distorted_image)
plt.title("Gaussian Noise - Medium")
plt.axis("off")
plt.show()

In [ ]:
# Save distortion settings
distortion_config_path = RESULTS_DIR / "distortion_config.csv"

distortion_summary_df.to_csv(distortion_config_path, index=False)

print("Distortion configuration saved to:")
print(distortion_config_path)


In [ ]:
# Evaluate ORB robustness under all distortion types and severity levels

orb = cv2.ORB_create(nfeatures=1500)
matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)

orb_distortion_results = []

for position, index in enumerate(selected_indices, start=1):
    clean_image, _ = dataset[index]

    clean_np = np.array(clean_image.convert("RGB"))
    clean_gray = cv2.cvtColor(clean_np, cv2.COLOR_RGB2GRAY)

    clean_keypoints, clean_descriptors = orb.detectAndCompute(
        clean_gray,
        None
    )

    for distortion_type, severity_dict in distortion_config.items():
        for severity_level in severity_dict:
            # Use an image-specific seed so Gaussian noise is reproducible,
            # while still being different for each dataset image
            image_seed = SEED + index

            distorted_image = apply_distortion(
                clean_image,
                distortion_type=distortion_type,
                severity_level=severity_level,
                seed=image_seed
            )

            distorted_np = np.array(distorted_image.convert("RGB"))
            distorted_gray = cv2.cvtColor(
                distorted_np,
                cv2.COLOR_RGB2GRAY
            )

            distorted_keypoints, distorted_descriptors = (
                orb.detectAndCompute(distorted_gray, None)
            )

            good_matches = []

            # Match descriptors only when both images contain valid descriptors
            if (
                clean_descriptors is not None
                and distorted_descriptors is not None
                and len(clean_descriptors) >= 2
                and len(distorted_descriptors) >= 2
            ):
                knn_matches = matcher.knnMatch(
                    clean_descriptors,
                    distorted_descriptors,
                    k=2
                )

                ratio_test_matches = []

                # Lowe's ratio test removes ambiguous descriptor matches
                for match_pair in knn_matches:
                    if len(match_pair) < 2:
                        continue

                    best_match, second_best_match = match_pair

                    if best_match.distance < 0.75 * second_best_match.distance:
                        ratio_test_matches.append(best_match)

                # Sort from strongest to weakest match
                ratio_test_matches = sorted(
                    ratio_test_matches,
                    key=lambda match: match.distance
                )

                # Enforce one-to-one matching:
                # each descriptor in the distorted image can be used only once
                used_distorted_descriptors = set()

                for match in ratio_test_matches:
                    if match.trainIdx not in used_distorted_descriptors:
                        good_matches.append(match)
                        used_distorted_descriptors.add(match.trainIdx)

            clean_keypoint_count = len(clean_keypoints)
            distorted_keypoint_count = len(distorted_keypoints)

            # Normalize by the smaller number of detected keypoints.
            # With one-to-one matching, this value is always between 0 and 1.
            denominator = min(
                clean_keypoint_count,
                distorted_keypoint_count
            )

            match_accuracy = (
                len(good_matches) / denominator
                if denominator > 0
                else 0.0
            )

            orb_distortion_results.append({
                "dataset_index": index,
                "distortion_type": distortion_type,
                "severity_level": severity_level,
                "parameter_value": distortion_config[
                    distortion_type
                ][severity_level],
                "clean_keypoints": clean_keypoint_count,
                "distorted_keypoints": distorted_keypoint_count,
                "good_matches": len(good_matches),
                "match_accuracy": match_accuracy
            })

    if position % 10 == 0:
        print(f"Processed {position}/{len(selected_indices)} images")

orb_distortion_df = pd.DataFrame(orb_distortion_results)

print("Finished ORB evaluation under distortions")
print(f"Number of result rows: {len(orb_distortion_df)}")

print(
    "Match accuracy range:",
    orb_distortion_df["match_accuracy"].min(),
    "to",
    orb_distortion_df["match_accuracy"].max()
)

display(orb_distortion_df.head())

In [ ]:
# Summarize ORB robustness by distortion type and severity level
orb_distortion_summary = (
    orb_distortion_df
    .groupby(
        ["distortion_type", "severity_level"],
        as_index=False
    )
    .agg(
        mean_match_accuracy=("match_accuracy", "mean"),
        std_match_accuracy=("match_accuracy", "std"),
        mean_good_matches=("good_matches", "mean"),
        mean_distorted_keypoints=("distorted_keypoints", "mean")
    )
)

severity_order = pd.CategoricalDtype(
    categories=["mild", "medium", "severe"],
    ordered=True
)

orb_distortion_summary["severity_level"] = (
    orb_distortion_summary["severity_level"]
    .astype(severity_order)
)

orb_distortion_summary = (
    orb_distortion_summary
    .sort_values(["distortion_type", "severity_level"])
    .reset_index(drop=True)
)

display(orb_distortion_summary)

In [ ]:
# Save detailed and summarized ORB distortion results
orb_distortion_df.to_csv(
    RESULTS_DIR / "orb_distortion_results.csv",
    index=False
)

orb_distortion_summary.to_csv(
    RESULTS_DIR / "orb_distortion_summary.csv",
    index=False
)

print("ORB distortion results saved successfully")


In [ ]:
# Plot ORB match accuracy across distortion types and severity levels

severity_order = ["mild", "medium", "severe"]

plt.figure(figsize=(10, 6))

for distortion_type in orb_distortion_summary["distortion_type"].unique():
    subset = orb_distortion_summary[
        orb_distortion_summary["distortion_type"] == distortion_type
    ].copy()

    subset["severity_level"] = pd.Categorical(
        subset["severity_level"],
        categories=severity_order,
        ordered=True
    )

    subset = subset.sort_values("severity_level")

    plt.plot(
        subset["severity_level"],
        subset["mean_match_accuracy"],
        marker="o",
        label=distortion_type
    )

plt.xlabel("Severity Level")
plt.ylabel("Mean ORB Match Accuracy")
plt.title("ORB Robustness Under Image Distortions")
plt.ylim(0, 1.05)
plt.grid(True)
plt.legend()
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "orb_match_accuracy_distortions.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Plot the average number of reliable ORB matches
plt.figure(figsize=(10, 6))

for distortion_type in orb_distortion_summary["distortion_type"].unique():
    subset = orb_distortion_summary[
        orb_distortion_summary["distortion_type"] == distortion_type
    ].copy()

    subset["severity_level"] = pd.Categorical(
        subset["severity_level"],
        categories=["mild", "medium", "severe"],
        ordered=True
    )

    subset = subset.sort_values("severity_level")

    plt.plot(
        subset["severity_level"],
        subset["mean_good_matches"],
        marker="o",
        label=distortion_type
    )

plt.xlabel("Severity Level")
plt.ylabel("Mean Number of Good Matches")
plt.title("ORB Reliable Matches Under Image Distortions")
plt.grid(True)
plt.legend()
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "orb_good_matches_distortions.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Load the pretrained semantic-segmentation model
import torch

from torchvision.models.segmentation import (
    deeplabv3_resnet50,
    DeepLabV3_ResNet50_Weights
)

segmentation_weights = DeepLabV3_ResNet50_Weights.DEFAULT

segmentation_model = deeplabv3_resnet50(
    weights=segmentation_weights
)

segmentation_model.eval()

segmentation_preprocess = segmentation_weights.transforms()

print("DeepLabV3 model loaded successfully")

In [ ]:
# Use GPU when available to speed up segmentation inference
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

segmentation_model = segmentation_model.to(device)

print("Using device:", device)

In [ ]:
# Test semantic segmentation on one distorted image
sample_index = selected_indices[0]
sample_image, sample_mask = dataset[sample_index]

test_distorted_image = apply_distortion(
    sample_image,
    distortion_type="gaussian_noise",
    severity_level="medium",
    seed=SEED + sample_index
)

input_tensor = (
    segmentation_preprocess(test_distorted_image)
    .unsqueeze(0)
    .to(device)
)

with torch.no_grad():
    output = segmentation_model(input_tensor)["out"]

predicted_mask = (
    output.argmax(dim=1)
    .squeeze(0)
    .cpu()
    .numpy()
    .astype(np.uint8)
)

predicted_mask_resized = cv2.resize(
    predicted_mask,
    sample_mask.size,
    interpolation=cv2.INTER_NEAREST
)

print("Ground-truth shape:", np.array(sample_mask).shape)
print("Prediction shape:", predicted_mask_resized.shape)
print("Predicted class IDs:", np.unique(predicted_mask_resized))

In [ ]:
# Evaluate semantic segmentation under all distortions and severity levels

segmentation_distortion_results = []
segmentation_per_class_results = []

segmentation_model.eval()

for position, index in enumerate(selected_indices, start=1):
    clean_image, ground_truth_pil = dataset[index]
    ground_truth = np.array(ground_truth_pil)

    for distortion_type, severity_dict in distortion_config.items():
        for severity_level in severity_dict:
            image_seed = SEED + index

            # Apply the selected distortion to the clean image
            distorted_image = apply_distortion(
                clean_image,
                distortion_type=distortion_type,
                severity_level=severity_level,
                seed=image_seed
            )

            # Apply the official DeepLabV3 preprocessing
            input_tensor = (
                segmentation_preprocess(distorted_image)
                .unsqueeze(0)
                .to(device)
            )

            # Run semantic-segmentation inference
            with torch.inference_mode():
                output = segmentation_model(input_tensor)["out"]

            prediction = (
                output.argmax(dim=1)
                .squeeze(0)
                .cpu()
                .numpy()
                .astype(np.uint8)
            )

            # Resize the prediction back to the original mask size
            prediction = cv2.resize(
                prediction,
                ground_truth_pil.size,
                interpolation=cv2.INTER_NEAREST
            )

            # Ignore pixels marked as void in PASCAL VOC
            valid_pixels = ground_truth != 255

            gt_valid = ground_truth[valid_pixels]
            pred_valid = prediction[valid_pixels]

            class_ids = np.union1d(
                np.unique(gt_valid),
                np.unique(pred_valid)
            )

            class_ious = {}

            for class_id in class_ids:
                intersection = np.logical_and(
                    gt_valid == class_id,
                    pred_valid == class_id
                ).sum()

                union = np.logical_or(
                    gt_valid == class_id,
                    pred_valid == class_id
                ).sum()

                if union > 0:
                    class_iou = intersection / union
                    class_ious[int(class_id)] = class_iou

                    segmentation_per_class_results.append({
                        "dataset_index": index,
                        "distortion_type": distortion_type,
                        "severity_level": severity_level,
                        "parameter_value": distortion_config[
                            distortion_type
                        ][severity_level],
                        "class_id": int(class_id),
                        "iou": class_iou
                    })

            mean_iou_all_classes = (
                np.mean(list(class_ious.values()))
                if class_ious
                else np.nan
            )

            foreground_ious = [
                iou
                for class_id, iou in class_ious.items()
                if class_id != 0
            ]

            mean_iou_foreground = (
                np.mean(foreground_ious)
                if foreground_ious
                else np.nan
            )

            segmentation_distortion_results.append({
                "dataset_index": index,
                "distortion_type": distortion_type,
                "severity_level": severity_level,
                "parameter_value": distortion_config[
                    distortion_type
                ][severity_level],
                "mean_iou_all_classes": mean_iou_all_classes,
                "mean_iou_foreground": mean_iou_foreground,
                "num_evaluated_classes": len(class_ious)
            })

    if position % 10 == 0:
        print(f"Processed {position}/{len(selected_indices)} images")

segmentation_distortion_df = pd.DataFrame(
    segmentation_distortion_results
)

segmentation_per_class_df = pd.DataFrame(
    segmentation_per_class_results
)

print("Finished semantic-segmentation evaluation under distortions")
print(
    "Number of image-condition rows:",
    len(segmentation_distortion_df)
)
print(
    "Number of per-class rows:",
    len(segmentation_per_class_df)
)

display(segmentation_distortion_df.head())

In [ ]:
# Summarize semantic-segmentation robustness by distortion and severity

segmentation_distortion_summary = (
    segmentation_distortion_df
    .groupby(
        ["distortion_type", "severity_level"],
        as_index=False
    )
    .agg(
        mean_iou_all_classes=("mean_iou_all_classes", "mean"),
        std_iou_all_classes=("mean_iou_all_classes", "std"),
        mean_iou_foreground=("mean_iou_foreground", "mean"),
        std_iou_foreground=("mean_iou_foreground", "std")
    )
)

severity_order = pd.CategoricalDtype(
    categories=["mild", "medium", "severe"],
    ordered=True
)

segmentation_distortion_summary["severity_level"] = (
    segmentation_distortion_summary["severity_level"]
    .astype(severity_order)
)

segmentation_distortion_summary = (
    segmentation_distortion_summary
    .sort_values(["distortion_type", "severity_level"])
    .reset_index(drop=True)
)

display(segmentation_distortion_summary)

In [ ]:
# Save detailed, per-class, and summarized segmentation distortion results
segmentation_distortion_df.to_csv(
    RESULTS_DIR / "segmentation_distortion_results.csv",
    index=False
)

segmentation_per_class_df.to_csv(
    RESULTS_DIR / "segmentation_distortion_per_class.csv",
    index=False
)

segmentation_distortion_summary.to_csv(
    RESULTS_DIR / "segmentation_distortion_summary.csv",
    index=False
)

print("Segmentation distortion results saved successfully")


In [ ]:
# Plot foreground mIoU across distortion types and severity levels

plt.figure(figsize=(10, 6))

for distortion_type in segmentation_distortion_summary["distortion_type"].unique():
    subset = segmentation_distortion_summary[
        segmentation_distortion_summary["distortion_type"] == distortion_type
    ].copy()

    subset["severity_level"] = pd.Categorical(
        subset["severity_level"],
        categories=["mild", "medium", "severe"],
        ordered=True
    )

    subset = subset.sort_values("severity_level")

    plt.plot(
        subset["severity_level"],
        subset["mean_iou_foreground"],
        marker="o",
        label=distortion_type
    )

plt.xlabel("Severity Level")
plt.ylabel("Mean Foreground IoU")
plt.title("Semantic Segmentation Robustness Under Image Distortions")
plt.ylim(0, 1.0)
plt.grid(True)
plt.legend()
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "segmentation_foreground_iou_distortions.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Load the pretrained YOLO object-detection model
from ultralytics import YOLO

detection_model = YOLO("yolo11n.pt")

print("YOLO object-detection model loaded successfully")

In [ ]:
# Prepare PASCAL VOC detection annotations and class-name mapping
import xml.etree.ElementTree as ET

voc_root = DATA_DIR / "VOCdevkit" / "VOC2012"
annotations_dir = voc_root / "Annotations"

YOLO_TO_VOC_CLASS_NAMES = {
    "airplane": "aeroplane",
    "motorcycle": "motorbike",
    "couch": "sofa",
    "tv": "tvmonitor",
    "potted plant": "pottedplant",
    "dining table": "diningtable"
}

def normalize_to_voc_class_name(class_name):
    return YOLO_TO_VOC_CLASS_NAMES.get(class_name, class_name)

print("Detection annotation setup completed")

In [ ]:
# Compute IoU between two bounding boxes in [xmin, ymin, xmax, ymax] format
def compute_box_iou(box_a, box_b):
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    intersection_width = max(0, x2 - x1)
    intersection_height = max(0, y2 - y1)
    intersection_area = intersection_width * intersection_height

    area_a = max(0, box_a[2] - box_a[0]) * max(0, box_a[3] - box_a[1])
    area_b = max(0, box_b[2] - box_b[0]) * max(0, box_b[3] - box_b[1])

    union_area = area_a + area_b - intersection_area

    return intersection_area / union_area if union_area > 0 else 0.0

In [ ]:
# Test YOLO on four distorted sample images before the full evaluation

test_indices = selected_indices[:4]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for axis, index in zip(axes, test_indices):
    image, _ = dataset[index]

    distorted_image = apply_distortion(
        image,
        distortion_type="gaussian_noise",
        severity_level="medium",
        seed=SEED + index
    )

    result = detection_model.predict(
        source=distorted_image,
        conf=0.25,
        verbose=False
    )[0]

    num_detections = len(result.boxes)

    if num_detections > 0:
        annotated_image = result.plot()
        annotated_image = cv2.cvtColor(
            annotated_image,
            cv2.COLOR_BGR2RGB
        )

        axis.imshow(annotated_image)
    else:
        axis.imshow(distorted_image)

    axis.set_title(
        f"Dataset Index {index}\nDetections: {num_detections}"
    )
    axis.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Read PASCAL VOC ground-truth boxes from an XML annotation
def read_voc_annotation(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    ground_truth_objects = []

    for object_element in root.findall("object"):
        class_name = object_element.find("name").text

        bounding_box = object_element.find("bndbox")

        box = [
            float(bounding_box.find("xmin").text),
            float(bounding_box.find("ymin").text),
            float(bounding_box.find("xmax").text),
            float(bounding_box.find("ymax").text)
        ]

        ground_truth_objects.append({
            "class_name": class_name,
            "box": box
        })

    return ground_truth_objects


# Match predictions to ground truth using class equality and IoU >= 0.5
def evaluate_detections(
    ground_truth_objects,
    predicted_objects,
    iou_threshold=0.5
):
    candidate_matches = []

    for prediction_index, prediction in enumerate(predicted_objects):
        for ground_truth_index, ground_truth in enumerate(
            ground_truth_objects
        ):
            if (
                prediction["class_name"]
                != ground_truth["class_name"]
            ):
                continue

            iou = compute_box_iou(
                prediction["box"],
                ground_truth["box"]
            )

            if iou >= iou_threshold:
                candidate_matches.append({
                    "prediction_index": prediction_index,
                    "ground_truth_index": ground_truth_index,
                    "iou": iou
                })

    # Highest-IoU matches are selected first
    candidate_matches.sort(
        key=lambda match: match["iou"],
        reverse=True
    )

    matched_predictions = set()
    matched_ground_truths = set()
    matched_ious = []

    for match in candidate_matches:
        prediction_index = match["prediction_index"]
        ground_truth_index = match["ground_truth_index"]

        if prediction_index in matched_predictions:
            continue

        if ground_truth_index in matched_ground_truths:
            continue

        matched_predictions.add(prediction_index)
        matched_ground_truths.add(ground_truth_index)
        matched_ious.append(match["iou"])

    true_positives = len(matched_ious)
    false_positives = (
        len(predicted_objects) - true_positives
    )
    false_negatives = (
        len(ground_truth_objects) - true_positives
    )

    return (
        true_positives,
        false_positives,
        false_negatives,
        matched_ious
    )

In [ ]:
# Evaluate YOLO object detection under all distortions

detection_distortion_results = []

for position, index in enumerate(selected_indices, start=1):
    clean_image, _ = dataset[index]

    image_path = Path(dataset.images[index])
    image_id = image_path.stem
    xml_path = annotations_dir / f"{image_id}.xml"

    ground_truth_objects = read_voc_annotation(xml_path)

    for distortion_type, severity_dict in distortion_config.items():
        for severity_level, parameter_value in severity_dict.items():

            distorted_image = apply_distortion(
                clean_image,
                distortion_type=distortion_type,
                severity_level=severity_level,
                seed=SEED + index
            )

            result = detection_model.predict(
                source=distorted_image,
                conf=0.25,
                verbose=False
            )[0]

            predicted_objects = []

            if len(result.boxes) > 0:
                predicted_boxes = (
                    result.boxes.xyxy
                    .cpu()
                    .numpy()
                )

                predicted_classes = (
                    result.boxes.cls
                    .cpu()
                    .numpy()
                    .astype(int)
                )

                predicted_confidences = (
                    result.boxes.conf
                    .cpu()
                    .numpy()
                )

                for box, class_id, confidence in zip(
                    predicted_boxes,
                    predicted_classes,
                    predicted_confidences
                ):
                    yolo_class_name = result.names[class_id]

                    voc_class_name = normalize_to_voc_class_name(
                        yolo_class_name
                    )

                    predicted_objects.append({
                        "class_name": voc_class_name,
                        "box": box.tolist(),
                        "confidence": float(confidence)
                    })

            (
                true_positives,
                false_positives,
                false_negatives,
                matched_ious
            ) = evaluate_detections(
                ground_truth_objects,
                predicted_objects,
                iou_threshold=0.5
            )

            precision = (
                true_positives
                / (true_positives + false_positives)
                if true_positives + false_positives > 0
                else 0.0
            )

            recall = (
                true_positives
                / (true_positives + false_negatives)
                if true_positives + false_negatives > 0
                else 0.0
            )

            f1_score = (
                2 * precision * recall / (precision + recall)
                if precision + recall > 0
                else 0.0
            )

            mean_matched_iou = (
                np.mean(matched_ious)
                if matched_ious
                else 0.0
            )

            detection_distortion_results.append({
                "dataset_index": index,
                "image_id": image_id,
                "distortion_type": distortion_type,
                "severity_level": severity_level,
                "parameter_value": parameter_value,
                "num_ground_truth_objects": len(
                    ground_truth_objects
                ),
                "num_predictions": len(predicted_objects),
                "true_positives": true_positives,
                "false_positives": false_positives,
                "false_negatives": false_negatives,
                "precision": precision,
                "recall": recall,
                "f1_score": f1_score,
                "mean_matched_iou": mean_matched_iou
            })

    if position % 10 == 0:
        print(f"Processed {position}/100 images")

detection_distortion_df = pd.DataFrame(
    detection_distortion_results
)

print("Finished object-detection evaluation")
print("Number of rows:", len(detection_distortion_df))

display(detection_distortion_df.head())

In [ ]:
# Summarize YOLO object-detection robustness

detection_distortion_summary = (
    detection_distortion_df
    .groupby(
        ["distortion_type", "severity_level"],
        as_index=False
    )
    .agg(
        total_true_positives=("true_positives", "sum"),
        total_false_positives=("false_positives", "sum"),
        total_false_negatives=("false_negatives", "sum"),
        mean_precision_per_image=("precision", "mean"),
        mean_recall_per_image=("recall", "mean"),
        mean_f1_per_image=("f1_score", "mean"),
        mean_matched_iou=("mean_matched_iou", "mean"),
        mean_num_predictions=("num_predictions", "mean")
    )
)

# Compute global precision, recall, and F1 from total TP, FP, and FN
detection_distortion_summary["global_precision"] = (
    detection_distortion_summary["total_true_positives"]
    / (
        detection_distortion_summary["total_true_positives"]
        + detection_distortion_summary["total_false_positives"]
    )
)

detection_distortion_summary["global_recall"] = (
    detection_distortion_summary["total_true_positives"]
    / (
        detection_distortion_summary["total_true_positives"]
        + detection_distortion_summary["total_false_negatives"]
    )
)

detection_distortion_summary["global_f1"] = (
    2
    * detection_distortion_summary["global_precision"]
    * detection_distortion_summary["global_recall"]
    / (
        detection_distortion_summary["global_precision"]
        + detection_distortion_summary["global_recall"]
    )
)

severity_order = pd.CategoricalDtype(
    categories=["mild", "medium", "severe"],
    ordered=True
)

detection_distortion_summary["severity_level"] = (
    detection_distortion_summary["severity_level"]
    .astype(severity_order)
)

detection_distortion_summary = (
    detection_distortion_summary
    .sort_values(["distortion_type", "severity_level"])
    .reset_index(drop=True)
)

display(
    detection_distortion_summary[
        [
            "distortion_type",
            "severity_level",
            "global_precision",
            "global_recall",
            "global_f1",
            "mean_matched_iou",
            "mean_num_predictions",
            "total_true_positives",
            "total_false_positives",
            "total_false_negatives"
        ]
    ]
)

In [ ]:
# Mean matched IoU only for images with at least one true-positive match

matched_iou_only_summary = (
    detection_distortion_df[
        detection_distortion_df["true_positives"] > 0
    ]
    .groupby(
        ["distortion_type", "severity_level"],
        as_index=False
    )
    .agg(
        mean_iou_when_matched=("mean_matched_iou", "mean"),
        images_with_matches=("dataset_index", "count")
    )
)

matched_iou_only_summary["severity_level"] = (
    matched_iou_only_summary["severity_level"]
    .astype(severity_order)
)

matched_iou_only_summary = (
    matched_iou_only_summary
    .sort_values(["distortion_type", "severity_level"])
    .reset_index(drop=True)
)

detection_distortion_summary = (
    detection_distortion_summary
    .merge(
        matched_iou_only_summary,
        on=["distortion_type", "severity_level"],
        how="left"
    )
)

display(
    detection_distortion_summary[
        [
            "distortion_type",
            "severity_level",
            "global_precision",
            "global_recall",
            "global_f1",
            "mean_iou_when_matched",
            "images_with_matches"
        ]
    ]
)

In [ ]:
# Save detailed and summarized object-detection results
detection_distortion_df.to_csv(
    RESULTS_DIR / "detection_distortion_results.csv",
    index=False
)

detection_distortion_summary.to_csv(
    RESULTS_DIR / "detection_distortion_summary.csv",
    index=False
)

print("Detection distortion results saved successfully")


In [ ]:
# Plot global F1 across distortion types and severity levels

plt.figure(figsize=(10, 6))

for distortion_type in detection_distortion_summary["distortion_type"].unique():
    subset = detection_distortion_summary[
        detection_distortion_summary["distortion_type"] == distortion_type
    ].copy()

    subset["severity_level"] = pd.Categorical(
        subset["severity_level"],
        categories=["mild", "medium", "severe"],
        ordered=True
    )

    subset = subset.sort_values("severity_level")

    plt.plot(
        subset["severity_level"],
        subset["global_f1"],
        marker="o",
        label=distortion_type
    )

plt.xlabel("Severity Level")
plt.ylabel("Global F1 Score")
plt.title("Object Detection Robustness Under Image Distortions")
plt.ylim(0, 1.0)
plt.grid(True)
plt.legend()
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "detection_global_f1_distortions.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()